# Self/other β reliability — genuinely-semantic neurons only
Same split-half reliability analysis as `02_beta_reliability.ipynb` (Plot 3c ranked bar, Plot 3d violin), but restricted to neurons whose `unique_semantic` (vs. all controls) is significant in **both** self and other conditions in the family-decomposition variance partitioning (notebook 07) — i.e. neurons whose semantic encoding survives controlling for lexical/syntactic/acoustic confounds in both conditions, not just neurons significant under the simpler semantic-alone model.
gpt2-xl L36, pc100, worddur window, xcirc null.

In [ ]:
import os, glob, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats as spstats
import seaborn as sns

plt.rcParams.update({'font.size': 11, 'axes.spines.top': False,
                     'axes.spines.right': False, 'figure.dpi': 130})

VP_DIR   = '/scratch/aniluchavez/ConvoDATAS/VPResults/gpt2-xl_ctx200_worddur_xcirc/pc100'
SEM_DIR  = '/scratch/aniluchavez/ConvoDATAS/SemanticGLM/gpt2-xl_ctx200_worddur_xcirc/pc100'
FIG_DIR  = '../figures'
LAYER    = 36
REGIONS  = ['hippocampus', 'ACC']
os.makedirs(FIG_DIR, exist_ok=True)

# ── VP family-decomposition significance (same loader as notebook 07) ───────
rows = []
for f in sorted(glob.glob(os.path.join(VP_DIR, 'PTY*_L36_VP.pkl'))):
    rows.append(pickle.load(open(f, 'rb')))
vp = pd.concat(rows, ignore_index=True)
print(f'VP: {len(vp)} rows, {vp["patient"].nunique()} patients')

# Neurons where unique_semantic (vs all controls) is significant in BOTH conditions
sig_self  = vp[(vp['condition'] == 'self')  & vp['significant']][['patient', 'region', 'neuron_idx']]
sig_other = vp[(vp['condition'] == 'other') & vp['significant']][['patient', 'region', 'neuron_idx']]
sig_both = pd.merge(sig_self, sig_other, on=['patient', 'region', 'neuron_idx'])
sig_both_set = set(map(tuple, sig_both.values))
print(f'Neurons significant (unique_semantic) in both self AND other: {len(sig_both_set)}')
for region in REGIONS:
    n_region = sum(1 for p, r, n in sig_both_set if r == region)
    n_total  = vp[(vp['region'] == region) & (vp['condition'] == 'self')]['neuron_idx'].count()
    print(f'  {region}: {n_region} / {n_total}')

In [ ]:
# ── Load split-half reliability data (semantic_glm.py --reliability output) ──
def load_patient_reliability(sem_dir, layer):
    rows = []
    for f in sorted(glob.glob(os.path.join(sem_dir, f'*_L{layer:02d}_sem.pkl'))):
        pid = os.path.basename(f).split(f'_L{layer:02d}')[0]
        obj = pickle.load(open(f, 'rb'))
        if not (isinstance(obj, dict) and 'reliability' in obj):
            continue
        for region, neuron_list in obj['reliability'].items():
            for r in neuron_list:
                rows.append({
                    'patient': pid, 'region': region, 'neuron_idx': r['neuron'],
                    'r_cross': r['r_cross'],
                    'self_reliability_mean': r['self_reliability_mean'],
                    'other_reliability_mean': r['other_reliability_mean'],
                    'ceil_mean': r['ceil_mean'],
                    'null_distribution': np.asarray(r.get('null_distribution', [])),
                })
    return pd.DataFrame(rows)

rel_df = load_patient_reliability(SEM_DIR, LAYER)
print(f'Reliability: {len(rel_df)} neuron rows, {rel_df["patient"].nunique()} patients')

# Filter to the genuinely-semantic population (significant unique_semantic, both conditions)
rel_df['key'] = list(zip(rel_df['patient'], rel_df['region'], rel_df['neuron_idx']))
rel_sig = rel_df[rel_df['key'].isin(sig_both_set)].drop(columns='key').reset_index(drop=True)
print(f'After filtering to unique_semantic-significant (both conditions): {len(rel_sig)} neurons')
for region in REGIONS:
    print(f'  {region}: {(rel_sig["region"] == region).sum()}')
rel_sig.head()

## Plot 1 — ranked bar of r_cross (self β vs other β, split-half), genuinely-semantic neurons only

In [ ]:
df_plot = rel_sig.dropna(subset=['r_cross'])
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, (region, color) in zip(axes, [('hippocampus', 'coral'), ('ACC', 'mediumseagreen')]):
    df_r = df_plot[df_plot['region'] == region].copy()
    if df_r.empty:
        ax.set_visible(False); continue
    df_r = df_r.sort_values('r_cross', ascending=False).reset_index(drop=True)
    df_r['rank'] = df_r.index + 1
    sns.barplot(data=df_r, x='rank', y='r_cross', color=color, ax=ax)
    ax.axhline(0, color='black', linewidth=1)
    ax.set_title(f'Self-Other Beta Correlation (split-half r_cross) — {region}\n'
                 f'gpt2-xl_ctx200_worddur_xcirc  L{LAYER}  (unique_semantic sig. both conditions, n={len(df_r)})')
    ax.set_xlabel('Neuron rank')
    ax.set_ylabel('r_cross (self β vs other β, full-data fit)')
    ax.tick_params(axis='x', labelbottom=False)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/08_beta_corr_bar_sig_semantic_L{LAYER}.pdf', bbox_inches='tight')
plt.show()

## Plot 2 — Speaking / Listening / Cross correlation violin, genuinely-semantic neurons only

In [ ]:
metric_labels = {
    'self_reliability_mean':  'Speaking',
    'other_reliability_mean': 'Listening',
    'r_cross':                'Cross correlation',
}
order   = ['Speaking', 'Listening', 'Cross correlation']
palette = {'Speaking': '#4C78A8', 'Listening': '#F58518', 'Cross correlation': '#54A24B'}

violin_df = rel_sig.melt(
    id_vars=['patient', 'region', 'neuron_idx'],
    value_vars=['self_reliability_mean', 'other_reliability_mean', 'r_cross'],
    var_name='metric', value_name='value',
)
violin_df['metric'] = violin_df['metric'].map(metric_labels)
violin_df = violin_df.dropna(subset=['value'])

regions_present = [r for r in REGIONS if (rel_sig['region'] == r).any()]
fig, axes = plt.subplots(1, len(regions_present),
                          figsize=(6.5 * len(regions_present), 5.5), sharey=True)
if len(regions_present) == 1:
    axes = [axes]
for ax, region in zip(axes, regions_present):
    rdf = violin_df[violin_df['region'] == region]
    n_neurons = (rel_sig['region'] == region).sum()
    sns.violinplot(data=rdf, x='metric', y='value', order=order, hue='metric',
                    palette=palette, legend=False, inner='quartile', cut=0, linewidth=1, ax=ax)
    sns.stripplot(data=rdf, x='metric', y='value', order=order,
                   color='black', alpha=0.3, size=3, jitter=0.18, ax=ax)
    ax.axhline(0, color='black', linewidth=1)
    ax.set_title(f'{region} reliability (n={n_neurons})')
    ax.set_xlabel('')
    ax.set_ylabel('Reliability / correlation')
    ax.tick_params(axis='x', rotation=20)
fig.suptitle('Speaking, listening, and cross beta correlation (split-half)\n'
             f'gpt2-xl_ctx200_worddur_xcirc  L{LAYER}  —  unique_semantic significant in both conditions', y=1.04)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/08_three_violin_sig_semantic_L{LAYER}.pdf', bbox_inches='tight')
plt.show()

# ── Significance tests: is Cross correlation above null, and below each ceiling? ──
print()
for region in regions_present:
    rdf_r = rel_sig[rel_sig['region'] == region].dropna(subset=['r_cross'])
    if rdf_r.empty:
        continue
    print(f'=== {region} (n={len(rdf_r)} neurons) ===')

    nulls = [nd for nd in rdf_r['null_distribution'] if nd is not None and len(nd) > 0]
    if nulls:
        min_n = min(len(nd) for nd in nulls)
        null_mat = np.vstack([nd[:min_n] for nd in nulls])
        R_null = null_mat.mean(axis=0)
        R_obs  = rdf_r['r_cross'].mean()
        p_null = (np.sum(R_null >= R_obs) + 1) / (len(R_null) + 1)
        print(f'  above null:  R_obs={R_obs:.3f}  null={R_null.mean():.3f}\u00b1{R_null.std():.3f}  '
              f'p(one-sided)={p_null:.3g}  (n_null={min_n})')
    else:
        print('  above null:  no null distributions available')

    for ceiling_col, label in [('self_reliability_mean', 'Speaking'),
                                ('other_reliability_mean', 'Listening')]:
        sub = rdf_r.dropna(subset=[ceiling_col])
        diff = sub['r_cross'] - sub[ceiling_col]
        t, p_two = spstats.ttest_1samp(diff, 0)
        p_one = p_two / 2 if t < 0 else 1 - p_two / 2
        pct_below = 100 * (diff < 0).mean()
        print(f'  below {label:9s} ceiling:  {pct_below:.1f}% below, '
              f'mean gap={-diff.mean():.3f}  t={t:.2f}  p(one-sided)={p_one:.3g}  n={len(sub)}')